# K-Means Clustering Pipeline
Builds clusters used by the Hybrid Recommendation System.


## 1. Imports


In [ ]:
import os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import RobustScaler, LabelEncoder, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
print('OK')


## 2. Paths


In [ ]:
NOTEBOOK_DIR = Path(os.getcwd())
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]
DATA_PATH    = PROJECT_ROOT / 'data' / 'processed' / 'cleaned_all_egypt.csv'
OUTPUT_DATA  = PROJECT_ROOT / 'data' / 'processed' / 'clustered_all_egypt.csv'
MODEL_DIR    = PROJECT_ROOT / 'models' / 'clustering'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Data :', DATA_PATH)
print('Models:', MODEL_DIR)


## 3. Load Dataset


In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
df.head(3)


## 4. Preprocessing


In [ ]:
FEATURE_COLS     = ['price', 'size', 'bedrooms', 'bathrooms',
                    'latitude', 'longitude', 'category', 'property_type']
NUMERICAL_COLS   = ['price', 'size', 'bedrooms', 'bathrooms',
                    'latitude', 'longitude']
CATEGORICAL_COLS = ['category', 'property_type']

df_feat = df[FEATURE_COLS].copy()

# Impute missing with median
for col in ['bedrooms', 'bathrooms']:
    df_feat[col] = df_feat[col].fillna(df_feat[col].median())

print('Nulls after imputation:')
print(df_feat.isnull().sum())


## 5. Feature Engineering


In [ ]:
# bed_bath_ratio: proxy for property layout
safe_baths = df_feat['bathrooms'].replace(0, 1)
df_feat['bed_bath_ratio'] = (df_feat['bedrooms'] / safe_baths).clip(upper=5)

# Log-transform skewed features
for col in ['price', 'size']:
    df_feat[f'log_{col}'] = np.log1p(df_feat[col])

LOG_COLS = ['log_price', 'log_size']
print('Engineered features:', ['bed_bath_ratio'] + LOG_COLS)


## 6. OneHotEncoding + Scaling


In [ ]:
# OneHotEncode categorical columns
label_encoders = {}
ohe_matrices   = []
ohe_col_names  = []

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    le.fit(df_feat[col].astype(str))
    label_encoders[col] = le

    ohe  = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    mat  = ohe.fit_transform(df_feat[[col]].astype(str))
    names = [f'{col}_{c}' for c in ohe.categories_[0]]
    ohe_matrices.append(mat)
    ohe_col_names.extend(names)

ohe_block = np.hstack(ohe_matrices)

# Build numerical block and scale
NUMERICAL_FINAL = LOG_COLS + ['bedrooms', 'bathrooms', 'bed_bath_ratio']
X_num   = df_feat[NUMERICAL_FINAL].values
scaler  = RobustScaler()
X_num_s = scaler.fit_transform(X_num)

# Final feature matrix
X = np.hstack([X_num_s, ohe_block])
FINAL_FEATURES = NUMERICAL_FINAL + ohe_col_names

print(f'Feature matrix : {X.shape}')
print(f'Numerical ({len(NUMERICAL_FINAL)}): {NUMERICAL_FINAL}')
print(f'OHE      ({len(ohe_col_names)}): {len(ohe_col_names)} columns')


## 7. Find Optimal K  (Elbow + Silhouette)


In [ ]:
K_RANGE = range(4, 21)
ks, inertias, sil_scores, db_scores = [], [], [], []

print(f'  {"K":>3}  {"Inertia":>14}  {"Silhouette":>12}  {"Davies-Bouldin":>15}')
print('  ' + '-' * 50)

for k in K_RANGE:
    km  = KMeans(n_clusters=k, init='k-means++', n_init=10,
                 max_iter=300, random_state=RANDOM_STATE)
    lbl = km.fit_predict(X)
    sil = silhouette_score(X, lbl, sample_size=10_000, random_state=RANDOM_STATE)
    db  = davies_bouldin_score(X, lbl)
    ks.append(k); inertias.append(km.inertia_)
    sil_scores.append(sil); db_scores.append(db)
    print(f'  {k:>3}  {km.inertia_:>14,.1f}  {sil:>12.4f}  {db:>15.4f}')


In [ ]:
# Elbow + Silhouette plots (side by side)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.plot(ks, inertias, marker='o', color='steelblue')
ax1.set_title('Elbow Curve'); ax1.set_xlabel('K'); ax1.set_ylabel('Inertia')
ax1.set_xticks(ks); ax1.grid(True, linestyle='--', alpha=0.6)

ax2.plot(ks, sil_scores, marker='s', color='coral')
ax2.set_title('Silhouette Score'); ax2.set_xlabel('K'); ax2.set_ylabel('Score')
ax2.set_xticks(ks); ax2.grid(True, linestyle='--', alpha=0.6)

plt.tight_layout(); plt.show()


In [ ]:
# Select best K: highest silhouette, balanced clusters (no cluster < 1%)
N = len(df)
km_cache = {}
for k in ks:
    km_tmp = KMeans(n_clusters=k, init='k-means++', n_init=10,
                    max_iter=300, random_state=RANDOM_STATE)
    km_tmp.fit(X)
    km_cache[k] = km_tmp

def balanced(k):
    sizes = np.bincount(km_cache[k].labels_)
    return sizes.min() / N * 100 >= 1.0

# Filter balanced, pick best silhouette
candidates = [(sil, k) for sil, k in zip(sil_scores, ks) if balanced(k)]
if not candidates:
    candidates = list(zip(sil_scores, ks))

OPTIMAL_K = max(candidates)[1]
print(f'Selected OPTIMAL_K = {OPTIMAL_K}')


## 8. Train Final K-Means Model


In [ ]:
kmeans = KMeans(
    n_clusters=OPTIMAL_K, init='k-means++',
    n_init=20, max_iter=500, random_state=RANDOM_STATE
)
cluster_labels = kmeans.fit_predict(X)
df['cluster']  = cluster_labels

final_sil = silhouette_score(X, cluster_labels,
                             sample_size=10_000, random_state=RANDOM_STATE)
final_db  = davies_bouldin_score(X, cluster_labels)

print(f'Final K              : {OPTIMAL_K}')
print(f'Silhouette Score     : {final_sil:.4f}')
print(f'Davies-Bouldin Index : {final_db:.4f}')
print(f'Inertia              : {kmeans.inertia_:,.2f}')
print(f'Iterations           : {kmeans.n_iter_}')


## 9. Cluster Size Summary


In [ ]:
sizes = df['cluster'].value_counts().sort_index()
print(f'Number of clusters : {OPTIMAL_K}')
print()
for cid, cnt in sizes.items():
    print(f'  Cluster {cid}: {cnt:>6,}  ({cnt/len(df)*100:.1f}%)')


## 10. Save Artifacts


In [ ]:
def save(obj, path, label):
    joblib.dump(obj, path)
    print(f'  {label:<40} {path.stat().st_size/1024:.1f} KB')

print(f'Saving to: {MODEL_DIR}')
save(kmeans,         MODEL_DIR / 'kmeans_model.joblib',   'kmeans_model.joblib')
save(scaler,         MODEL_DIR / 'scaler.joblib',         'scaler.joblib')
save(label_encoders, MODEL_DIR / 'label_encoders.joblib', 'label_encoders.joblib')
save(FINAL_FEATURES, MODEL_DIR / 'feature_names.joblib',  'feature_names.joblib')

# property_id -> cluster mapping
id_cluster = df[['id', 'cluster']]
save(id_cluster,     MODEL_DIR / 'id_cluster_map.joblib', 'id_cluster_map.joblib')

# Clustered dataset
df.to_csv(OUTPUT_DATA, index=False)
print(f'  {"clustered_all_egypt.csv":<40} {OUTPUT_DATA.stat().st_size/1024/1024:.2f} MB')
print()
print('All artifacts saved.')


## 11. Verify Artifacts


In [ ]:
lk = joblib.load(MODEL_DIR / 'kmeans_model.joblib')
ls = joblib.load(MODEL_DIR / 'scaler.joblib')
lf = joblib.load(MODEL_DIR / 'feature_names.joblib')
lm = joblib.load(MODEL_DIR / 'id_cluster_map.joblib')
dc = pd.read_csv(OUTPUT_DATA)

print(f'KMeans n_clusters  : {lk.n_clusters}')
print(f'Scaler type        : {type(ls).__name__}')
print(f'Feature count      : {len(lf)}')
print(f'ID-cluster rows    : {len(lm)}')
print(f'Clustered CSV shape: {dc.shape}')
print(f'"cluster" column   : {"cluster" in dc.columns}')
